# Publications markdown generator for academicpages

Takes a set of bibtex of publications and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). 

The core python code is also in `pubsFromBibs.py`. 
Run either from the `markdown_generator` folder after replacing updating the publist dictionary with:
* bib file names
* specific venue keys based on your bib file preferences
* any specific pre-text for specific files
* Collection Name (future feature)

TODO: Make this work with other databases of citations, 
TODO: Merge this with the existing TSV parsing solution

In [1]:
from pybtex.database.input import bibtex
import pybtex.database.input.bibtex 
from time import strptime
import string
import html
import os
import re
import urllib.parse
import datetime


In [7]:
#todo: incorporate different collection types rather than a catch all publications, requires other changes to template
publist = {
    #"proceeding": {
        #"file" : "proceedings.bib",
        #"venuekey": "booktitle",
        #"venue-pretext": "In the proceedings of ",
        #"collection" : {"name":"publications",
                        #"permalink":"/publication/"}
        
    #},
    "journal":{
        "file": "publications.bib",
        "venuekey" : "journal",
        "venue-pretext" : "",
        "collection" : {"name":"publications",
                        "permalink":"/publication/"}
    },
#    "preprint":{
#        "file": "preprints.bib",
#        "venuekey" : "note",
#        "venue-pretext" : "",
#        "collection" : {"name":"publications",
#                        "permalink":"/publication/"}
#    } 
}

In [2]:
html_escape_table = {
    "\&": "&amp;",
    '"': "&quot;",
    "'": "&apos;",
    "á": "&aacute;",
    "ä": "&auml;",
    "é": "&eacute;",
    "í": "&iacute;",
    "ó": "&oacute;",
    "ú": "&uacute;",
    "Á": "&Aacute;",
    "É": "&Eacute;",
    "Í": "&Iacute;",
    "Ó": "&Oacute;",
    "Ú": "&Uacute;",
    "ñ": "&ntilde;",
    "Ñ": "&Ntilde;",
    }

def html_escape(text):
    """Produce entities within text."""
    return "".join(html_escape_table.get(c,c) for c in text)

In [3]:
bibFile = "website.bib"
parser = bibtex.Parser()
today = datetime.date.today()
bibdata = parser.parse_file(bibFile)
for bib_key in bibdata.entries:
  sbib_key=bib_key.split("_")
  entry = bibdata.entries[bib_key]
  bib_type = entry.type
  fields = entry.fields
  try:

    if "research" in fields["keywords"] or "outreach" in fields["keywords"]:

      #fix the pub date
      year="1900"
      month= "01"
      day= "01"
      year = fields["year"]
      if "month" in fields.keys():
        if(len(fields["month"])<3):
          month = "0"+fields["month"]
          month = pub_month[-2:]
        elif(fields["month"] not in range(12)):
          tmnth = strptime(fields["month"][:3],'%b').tm_mon
          month = "{:02d}".format(tmnth)
        else:
          month = str(fields["month"])
      if "day" in fields.keys():
        day = str(fields["day"])

      date = year+"-"+month+"-"+day

      title=""
      title=html_escape(fields["title"].replace("{", "").replace("}","").replace("\\",""))
      #clean_title = fields["title"].replace("{", "").replace("}","").replace("\\","").replace(" ","-")


      #compute the author string
      author=""
      fa= bibdata.entries[bib_key].persons["author"][0]
      author=" ".join(n[0]+"." for n in fa.bibtex_first_names)+" "+fa.last_names[0]
      for i in range(1,len(bibdata.entries[bib_key].persons["author"])):
        aa = bibdata.entries[bib_key].persons["author"][i]
        author = author+", "+" ".join(n[0]+"." for n in aa.bibtex_first_names)+" "+aa.last_names[0]
      author=author+"."
      author=html_escape(author)

      #journal
      journal=""
      if "journal" in fields.keys():
        journal = html_escape(fields["journal"])
      else:
        journal= html_escape(fields["journaltitle"])
      

      paperurl=""
      if "doi" in fields.keys():
        paperurl = "https://doi.org/"+fields["doi"]
      elif "url" in fields.keys():
        paperurl = fields["url"]
      #paperurl=urllib.parse.quote_plus(paperurl)

      #abstract
      abstract=""
      if "abstract" in fields.keys():
        abstract = html_escape(fields["abstract"])
        abstract.replace("\%","%")

      #bibstring
      bib_string=""
      bib_string = entry.to_string("bibtex")

      html_filename= year + "_" + sbib_key[0] + "_" + sbib_key[2]
      md_filename= html_filename+".md"

      #YAML variables:
      md = "---\ntitle: \""   + title +'"\n'
      md = md + "collection: publications \npermalink: /publication/"+html_filename
      md += "\ndate: " + str(date)
      md += "\nauthor: '" + str(author) +"'"
      md += "\ncitation: '" + str(author) + " " + str(title) + ", " + str(journal) + " (" + str(year) + "). " + str(paperurl) + ".'"
      md += "\njournal: '" +  str(journal) +"'"
      md += "\npaperurl: '" + str(paperurl) +"'"
      if "research" in fields["keywords"]:
        md += "\ntype: '" + "research" +"'"
      if "outreach" in fields["keywords"]:
        md += "\ntype: '" + "outreach" +"'"
      md += "\n--- \n"

      ##Markdown description for individual page

      md += f"\n[Access paper here]({paperurl})"+"{:target=\"_blank\"}\n"

      md += "\n**Abstract**: " + str(abstract)

      md += "\n\nBibtex:\n"
      md += "``` \n " + bib_string + "```"

      md_filename = os.path.basename(md_filename)
      with open("../_publications/" + md_filename, 'w') as f:
                f.write(md)
      print(f'SUCESSFULLY PARSED {bib_key}')

    #here finishes the processing of articles
    if "preparation" in fields["keywords"]:

      #fix the pub date
      year=str(today.year)
      month= "01"
      day= "01"
      if "year" in fields.keys():
        year = fields["year"]
      if "month" in fields.keys():
        if(len(fields["month"])<3):
          month = "0"+fields["month"]
          month = pub_month[-2:]
        elif(fields["month"] not in range(12)):
          tmnth = strptime(fields["month"][:3],'%b').tm_mon
          month = "{:02d}".format(tmnth)
        else:
          month = str(fields["month"])
      if "day" in fields.keys():
        day = str(fields["day"])

      date = year+"-"+month+"-"+day

      title=""
      title=html_escape(fields["title"].replace("{", "").replace("}","").replace("\\",""))
      #clean_title = fields["title"].replace("{", "").replace("}","").replace("\\","").replace(" ","-")


      #compute the author string
      autor=""
      fa= bibdata.entries[bib_key].persons["author"][0]
      author=" ".join(n[0]+"." for n in fa.bibtex_first_names)+" "+fa.last_names[0]
      for i in range(1,len(bibdata.entries[bib_key].persons["author"])):
        aa = bibdata.entries[bib_key].persons["author"][i]
        author = author+", "+" ".join(n[0]+"." for n in aa.bibtex_first_names)+" "+aa.last_names[0]
      author=author+"."
      author=html_escape(author)

      #journal
      journal=""
      if "journal" in fields.keys():
        journal = html_escape(fields["journal"])
      elif "journaltitle" in fields.keys():
        journal= html_escape(fields["journaltitle"])
      elif "note" in fields.keys():
        journal= html_escape(fields["note"])

      #note
      note=""
      if "note" in fields.keys():
        journal = html_escape(fields["note"].replace("\%","%"))

      paperurl=""
      if "doi" in fields.keys():
        paperurl = "https://doi.org/"+fields["doi"]
      elif "url" in fields.keys():
        paperurl = fields["url"]
      #paperurl=urllib.parse.quote_plus(paperurl)

      #abstract
      abstract=""
      if "abstract" in fields.keys():
        abstract = html_escape(fields["abstract"])

      #bibstring
      fields["note"]="In preparation"
      fields["keywords"]=fields["keywords"].replace("preparation","")
      bib_string = entry.to_string("bibtex")

      html_filename= year + "_" + sbib_key[0] + "_" + sbib_key[1]
      md_filename= html_filename+".md"

      #YAML variables:
      md = "---\ntitle: \""   + title +'"\n'
      md = md + "collection: publications \npermalink: /publication/"+html_filename
      md += "\ndate: " + str(date)
      md += "\nauthor: '" + str(author) +"'"
      if len(paperurl) > 0:
        md += "\npaperurl: '" + str(paperurl) +"'"
        md += "\ncitation: '" + str(author) + " " + str(title) + " (preprint). " + str(paperurl) + ".'"
      else:
        md += "\ncitation: '" + str(author) + " " + str(title) + " (preprint). '" 
      md += "\njournal: '" +  str(journal) +"'"
      md += "\ntype: '" + "preprint" +"'"
      md += "\n--- \n"

      ##Markdown description for individual page
      if len(paperurl) > 0:
        md += f"\n[Preprint here]({paperurl})"+"{:target=\"_blank\"}\n"
      else:
        md +="Preprint not available yet.\n"

      if len(abstract) > 0:
        md += "\n**Abstract**: " + str(abstract)
      else:
        md +="Abstract not available."
      
   
      md += "\n\nBibtex:\n"
      md += "``` \n " + bib_string + "```"

      md_filename = os.path.basename(md_filename)
      with open("../_publications/" + md_filename, 'w') as f:
                f.write(md)
      print(f'SUCESSFULLY PARSED {bib_key}')


    #here it finishes the processing of preprints
    if "thesis" in fields["keywords"]:

      #fix the pub date
      year=str(today.year)
      month= "01"
      day= "01"
      if "year" in fields.keys():
        year = fields["year"]
      if "month" in fields.keys():
        if(len(fields["month"])<3):
          month = "0"+fields["month"]
          month = pub_month[-2:]
        elif(fields["month"] not in range(12)):
          tmnth = strptime(fields["month"][:3],'%b').tm_mon
          month = "{:02d}".format(tmnth)
        else:
          month = str(fields["month"])
      if "day" in fields.keys():
        day = str(fields["day"])

      date = year+"-"+month+"-"+day

      title=""
      title=html_escape(fields["title"].replace("{", "").replace("}","").replace("\\",""))
      #clean_title = fields["title"].replace("{", "").replace("}","").replace("\\","").replace(" ","-")


      #compute the author string
      autor=""
      fa= bibdata.entries[bib_key].persons["author"][0]
      author=" ".join(n[0]+"." for n in fa.bibtex_first_names)+" "+fa.last_names[0]
      for i in range(1,len(bibdata.entries[bib_key].persons["author"])):
        aa = bibdata.entries[bib_key].persons["author"][i]
        author = author+", "+" ".join(n[0]+"." for n in aa.bibtex_first_names)+" "+aa.last_names[0]
      author=author+"."
      author=html_escape(author)

      #journal
      journal=""
      if "type" in fields.keys() or "school" in fields.keys():
        journal = html_escape(fields["type"]) + ", " + html_escape(fields["school"]) + "."
 

      #note
      note=""
      if "note" in fields.keys():
        journal = html_escape(fields["note"].replace("\%","%"))

      paperurl=""
      if "doi" in fields.keys():
        paperurl = "https://doi.org/"+fields["doi"]
      elif "url" in fields.keys():
        paperurl = fields["url"]
      #paperurl=urllib.parse.quote_plus(paperurl)

      #abstract()
      abstract=""
      if "abstract" in fields.keys():
        abstract = html_escape(fields["abstract"])

      #bibstring
      fields["keywords"]=fields["keywords"].replace("thesis","")
      bib_string = entry.to_string("bibtex")

      html_filename= year + "_" + sbib_key[0] + "_" + sbib_key[2]
      md_filename= html_filename+".md"

      #YAML variables:
      md = "---\ntitle: \""   + title +'"\n'
      md = md + "collection: publications \npermalink: /publication/"+html_filename
      md += "\ndate: " + str(date)
      md += "\nauthor: '" + str(author) +"'"
      if len(paperurl) > 0:
        md += "\npaperurl: '" + str(paperurl) +"'"
      md += "\ncitation: '" + str(author) + " " + str(title) + ", " + str(journal) + ", " + str(year) + ". " + "'"
      md += "\njournal: '" +  str(journal) +"'"
      md += "\ntype: '" + "thesis" +"'"
      md += "\n--- \n"

      ##Markdown description for individual page
      if len(paperurl) > 0:
        md += f"\n[Download it here]({paperurl})"+"{:target=\"_blank\"}\n"
      else:
        md +="\nPreprint not available yet.\n"

      if len(abstract) > 0:
        md += "\n**Abstract**: " + str(abstract)
      else:
        md +="\nAbstract not available."
      
   
      md += "\n\nBibtex:\n"
      md += "``` \n " + bib_string + "```"

      md_filename = os.path.basename(md_filename)
      with open("../_publications/" + md_filename, 'w') as f:
                f.write(md)
      print(f'SUCESSFULLY PARSED {bib_key}')


      #here finishes the processing of articles


  except KeyError as e:
    print(f'WARNING Missing Expected Field {e} from entry {bib_key}: \"', fields["title"][:30],"..."*(len(fields['title'])>30),"\"")

  continue

SUCESSFULLY PARSED Montero_2018_RegularPolyhedra3
SUCESSFULLY PARSED MonteroPellicerToledo_ChiralExtensionsRegular_preprint
SUCESSFULLY PARSED Montero_2015_RealizacionesRegularesDe_Thesis
SUCESSFULLY PARSED Montero_2013_PoliedrosRegularesEn_Thesis
SUCESSFULLY PARSED Montero_2019_ChiralExtensionsToroids_PhDThesis
SUCESSFULLY PARSED Montero_2016_SimetriasLasMatematicas
SUCESSFULLY PARSED Montero_2016_SimetriasDePoligonos
SUCESSFULLY PARSED CollinsMontero_2021_EquivelarToroidsFew
SUCESSFULLY PARSED CunninghamMochanMontero_CayleyExtensionsManiplexes_preprint
SUCESSFULLY PARSED Montero_2021_SchlaefliSymbolChiral
SUCESSFULLY PARSED MonteroWeiss_2021_ProperLocallySpherical
SUCESSFULLY PARSED MonteroIvicWeiss_2021_LocallySphericalHypertopes
SUCESSFULLY PARSED MonteroPotocnik_VertexTransitiveGraphs_preprint
SUCESSFULLY PARSED HubardMochanMontero_2023_VoltageOperationsManiplexes
SUCESSFULLY PARSED HubardMochanMontero_SymmetriesVoltageOperations_preprint


In [145]:
entry=bibdata.entries["MonteroIvicWeiss_2021_LocallySphericalHypertopes"]
print(entry.to_string("bibtex"))

@Article{MonteroIvicWeiss_2021_LocallySphericalHypertopes,
    author = "Montero, Antonio and Weiss, Asia Ivić",
    title = "Locally spherical hypertopes from generalised cubes",
    doi = "10.26493/2590-9770.1354.b40",
    number = "3",
    pages = "Paper No. 3.07, 12",
    url = "https://doi.org/10.26493/2590-9770.1354.b40",
    volume = "4",
    abstract = "We show that every non-degenerate regular polytope can be used to construct a thin, residually-connected, chamber-transitive incidence geometry, i.e. a regular hypertope. These hypertopes are related to the semi-regular polyotopes with a tail-triangle Coxeter diagram constructed by Monson and Schulte. We discuss several interesting examples derived when this construction is applied to generalised cubes. In particular, we produce an example of a rank 5 finite locally spherical proper hypertope of hyperbolic type. No such examples were previously known.",
    fjournal = "The Art of Discrete and Applied Mathematics",
    journal = 

In [12]:
for pubsource in publist:
    parser = bibtex.Parser()
    bibdata = parser.parse_file(publist[pubsource]["file"])

    #loop through the individual references in a given bibtex file
    for bib_id in bibdata.entries: 
        #reset default date
        pub_year = "1900"
        pub_month = "01"
        pub_day = "01"
        
        b = bibdata.entries[bib_id].fields
        try:
            pub_year = b["year"][:4]

            #todo: this hack for month and day needs some cleanup
            if "month" in b.keys(): 
                if(len(b["month"])<3):
                    pub_month = "0"+b["month"]
                    pub_month = pub_month[-2:]
                elif(b["month"] not in range(12)):
                    tmnth = strptime(b["month"][:3],'%b').tm_mon   
                    pub_month = "{:02d}".format(tmnth) 
                else:
                    pub_month = str(b["month"])
            if "day" in b.keys(): 
                pub_day = str(b["day"])
            
            pub_date = pub_year+"-"+pub_month+"-"+pub_day
 
            clean_title = b["title"].replace("{", "").replace("}","").replace("\\","").replace(" ","-")

                
            url_slug = re.sub("\\[.*\\]|[^a-zA-Z0-9_-]", "", clean_title)
            url_slug = url_slug.replace("--","-")

            md_filename = pub_year+"_"+bib_id+".md"
            html_filename =  pub_year+"_"+bib_id

            #Build Citation from text
            citation = ""
            #citation authors - todo - add highlighting for primary author?
            for author in bibdata.entries[bib_id].persons["author"]:
                citation = citation+" "+author.first_names[0]+" "+author.last_names[0]+", "
            #citation title
            citation = citation + "\"" + html_escape(b["title"].replace("{", "").replace("}","").replace("\\","")) + ".\""
            #add venue logic depending on citation type
            venue = publist[pubsource]["venue-pretext"]+b[publist[pubsource]["venuekey"]].replace("{", "").replace("}","").replace("\\","")
            citation = citation + " " + html_escape(venue)
            citation = citation + ", " + pub_year + "."


            ## YAML variables
            md = "---\ntitle: \""   + html_escape(b["title"].replace("{", "").replace("}","").replace("\\","")) + '"\n'

            md += """collection: """ +  publist[pubsource]["collection"]["name"]

            md += """\npermalink: """ + publist[pubsource]["collection"]["permalink"]  + html_filename


            note = False
            if "note" in b.keys():
                if len(str(b["note"])) > 5:
                    md += "\nexcerpt: '" + html_escape(b["note"]) + "'"
                    note = True

            md += "\ndate: " + str(pub_date) 

            md += "\nvenue: '" + html_escape(venue) + "'"
            
            doi = False
            if "doi" in b.keys():
                if len(str(b["doi"])) > 5:
                    md += "\npaperurl: '" + "https://doi.org/"+b["doi"] + "'"
                    doi = True


            url = False
            if "url" in b.keys() and not doi:
                if len(str(b["url"])) > 5:
                    md += "\npaperurl: '" + b["url"] + "'"
                    url = True

            md += "\ncitation: '" + html_escape(citation) + "'"

            md += "\n--- \n"


            ## Markdown description for individual page
            if note:
                md += "\n" + html_escape(b["note"]) + "\n"
            if doi:
                md += "\n[Access paper here]( http://doi.org/" + b["doi"] + "){:target=\"_blank\"}\n"
            elif url:
                md += "\n[Access paper here](" + b["url"] + "){:target=\"_blank\"}\n" 

            md_filename = os.path.basename(md_filename)

            with open("../_publications/" + md_filename, 'w') as f:
                f.write(md)
            print(f'SUCESSFULLY PARSED {bib_id}: \"', b["title"][:60],"..."*(len(b['title'])>60),"\"")

        except KeyError as e:
            print(f'WARNING Missing Expected Field {e} from entry {bib_id}: \"', b["title"][:30],"..."*(len(b['title'])>30),"\"")

    continue


SUCESSFULLY PARSED Montero_2018_RegularPolyhedra3: " Regular polyhedra in the 3-torus  "
WARNING Missing Expected Field 'year' from entry MonteroPellicerToledo_ChiralExtensionsRegular_preprint: " Chiral extensions of regular t ... "
WARNING Missing Expected Field 'journal' from entry Montero_2015_RealizacionesRegularesDe_Thesis: " Realizaciones regulares de pol ... "
WARNING Missing Expected Field 'journal' from entry Montero_2013_PoliedrosRegularesEn_Thesis: " Poliedros regulares en el $3$- ... "
WARNING Missing Expected Field 'journal' from entry Montero_2019_ChiralExtensionsToroids_PhDThesis: " Chiral extensions of toroids  "
SUCESSFULLY PARSED Montero_2016_SimetriasLasMatematicas: " Simetrías: Las matemáticas detrás de la belleza \ifbool{en}{ ... "
SUCESSFULLY PARSED Montero_2016_SimetriasDePoligonos: " Simetrías de polígonos, poliedros y otras policosas \ifbool{ ... "
SUCESSFULLY PARSED CollinsMontero_2021_EquivelarToroidsFew: " Equivelar {Toroids} with {Few} {Flag}-{Orbits}  "
WA